# Phase 2 — RAG Pipeline: Ingestion, Chunking, Embeddings, Retrieval & Evaluation

This notebook implements Phase 2 of the RAG-Powered Document Assistant project:

1. Load & Inspect source documents
2. Chunking strategy
3. Embeddings & vector store (ChromaDB, persisted locally)
4. Retrieval & prompting (with citation-style grounding)
5. Evaluation against sample questions
6. Export / persistence check

**Assumed project structure** (relative to this notebook in `notebooks/`):

```
project_root/
├── data/
│   ├── raw/            # source PDFs live here (already populated)
│   └── vector_store/   # ChromaDB persistence directory (created by this notebook)
└── notebooks/
    └── phase2_rag_pipeline.ipynb
```


## 0. Setup & Dependencies

Install once (uncomment if needed), then import. We pin to widely-compatible packages:

- `pypdf` — PDF parsing (lightweight, no external system deps)
- `langchain` / `langchain-text-splitters` — chunking utilities
- `sentence-transformers` — local embedding model (`all-MiniLM-L6-v2`)
- `chromadb` — local, persistent vector store
- `pandas` — evaluation results table


In [1]:
# %pip install pypdf langchain langchain-text-splitters sentence-transformers chromadb pandas --quiet

import os
import glob
import time
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from IPython.display import display, Markdown

# ---- Project paths (relative to notebooks/ folder) ----
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "backend" / "data" / "raw"
VECTOR_STORE_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:     {PROJECT_ROOT}")
print(f"Raw data dir:     {RAW_DIR}")
print(f"Vector store dir: {VECTOR_STORE_DIR}")


f:\rag-assistant-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root:     F:\rag-assistant-project
Raw data dir:     F:\rag-assistant-project\backend\data\raw
Vector store dir: F:\rag-assistant-project\backend\data\vector_store


## 1. Load & Inspect Documents

We scan `data/raw/`, load every supported file, and collect stats (format, page count, parsing issues) so we can answer the required questions with real numbers rather than guesses.

In [2]:
SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}

def load_pdf(path: Path):
    """Load a PDF and return (list_of_page_texts, page_count, error)."""
    try:
        reader = PdfReader(str(path))
        pages = []
        for i, page in enumerate(reader.pages):
            try:
                text = page.extract_text() or ""
            except Exception as e:
                text = ""
                print(f"  [WARN] Could not extract text from page {i} of {path.name}: {e}")
            pages.append(text)
        return pages, len(reader.pages), None
    except Exception as e:
        return [], 0, str(e)


def load_text_file(path: Path):
    """Load a plain text / markdown file as a single 'page'."""
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
        return [text], 1, None
    except Exception as e:
        return [], 0, str(e)


def load_documents(raw_dir: Path):
    """
    Walk raw_dir, load every supported file, and return:
      - documents: list of dicts {source, format, page_texts, page_count}
      - inspection_log: list of dicts with per-file stats / issues
    """
    documents = []
    inspection_log = []

    all_files = sorted([p for p in raw_dir.glob("**/*") if p.is_file()])

    for path in all_files:
        ext = path.suffix.lower()
        if ext not in SUPPORTED_EXTENSIONS:
            inspection_log.append({
                "file": path.name, "format": ext or "unknown",
                "pages": 0, "status": "SKIPPED (unsupported format)"
            })
            continue

        if ext == ".pdf":
            pages, page_count, error = load_pdf(path)
        else:
            pages, page_count, error = load_text_file(path)

        if error:
            inspection_log.append({
                "file": path.name, "format": ext, "pages": 0,
                "status": f"ERROR: {error}"
            })
            continue

        empty_pages = sum(1 for p in pages if not p.strip())
        status = "OK"
        if empty_pages:
            status = f"OK ({empty_pages} page(s) returned no extractable text)"

        documents.append({
            "source": path.name,
            "format": ext,
            "page_texts": pages,
            "page_count": page_count,
        })
        inspection_log.append({
            "file": path.name, "format": ext,
            "pages": page_count, "status": status
        })

    return documents, inspection_log


documents, inspection_log = load_documents(RAW_DIR)
inspection_df = pd.DataFrame(inspection_log)
inspection_df


,file,format,pages,status
0,AI-info.pdf,.pdf,10,OK


In [3]:
# ---- Aggregate stats for the write-up below ----
total_docs = len(documents)
total_pages = sum(d["page_count"] for d in documents)
formats_found = sorted({d["format"] for d in documents})
parsing_issues = [row for row in inspection_log if row["status"] != "OK" and not row["status"].startswith("OK (")]
pages_with_warnings = [row for row in inspection_log if row["status"].startswith("OK (")]

summary_md = f"""
### Load & Inspect — Summary

- **Documents successfully loaded:** {total_docs}
- **Total pages parsed:** {total_pages}
- **Formats found in `data/raw/`:** {", ".join(formats_found) if formats_found else "none"}
- **Files with hard parsing errors:** {len(parsing_issues)}
- **Files with partial/empty-page extraction warnings:** {len(pages_with_warnings)}

{"No parsing issues detected." if not parsing_issues and not pages_with_warnings else
 "See the inspection table above for per-file details (rows with status != 'OK')."}
"""
display(Markdown(summary_md))



### Load & Inspect — Summary

- **Documents successfully loaded:** 1
- **Total pages parsed:** 10
- **Formats found in `data/raw/`:** .pdf
- **Files with hard parsing errors:** 0
- **Files with partial/empty-page extraction warnings:** 0

No parsing issues detected.


> **Answers (auto-generated above from the actual scan of `data/raw/`):**
> - *How many documents/pages?* → see "Documents successfully loaded" / "Total pages parsed" in the summary cell output.
> - *What formats?* → see "Formats found" above. This pipeline currently supports `.pdf`, `.txt`, `.md`; extend `SUPPORTED_EXTENSIONS` and add a loader function if you need `.docx`, `.html`, etc.
> - *Any parsing issues?* → check the inspection table; rows are flagged `ERROR` (file failed to open/parse) or `OK (n page(s) returned no extractable text)` (common with scanned/image-only PDF pages, which need OCR — out of scope for Phase 2).

## 2. Chunking Strategy

We use LangChain's `RecursiveCharacterTextSplitter`, which tries to split on paragraph → sentence → word boundaries (in that order) before falling back to a hard character cut. This keeps chunks semantically coherent instead of cutting mid-sentence.

**Chosen parameters:**

| Parameter | Value | Justification |
|---|---|---|
| `chunk_size` | 800 characters (~150–200 tokens) | Small enough that each chunk stays topically focused (improves embedding quality and retrieval precision), large enough to retain enough context for the LLM to answer without needing many chunks stitched together. This is the commonly recommended middle ground for general-purpose document Q&A (too small → fragmented, context-less snippets; too large → dilutes embedding relevance and wastes prompt tokens). |
| `chunk_overlap` | 120 characters (~15% of chunk size) | Prevents important sentences that straddle a chunk boundary from losing context on either side. 10–20% overlap is standard practice — enough to preserve continuity without excessive duplication/storage bloat. |
| `separators` | `["\n\n", "\n", ". ", " ", ""]` | Prioritizes paragraph breaks, then line breaks, then sentence boundaries, before falling back to raw character splitting — keeps chunks human-readable. |

These are reasonable defaults for prose-heavy PDFs; if your documents are very dense/technical (e.g. legal or tabular content) you may want a smaller `chunk_size` (e.g. 400–500) to keep chunks more precise.

In [4]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

def chunk_documents(documents):
    """
    Turn loaded documents into a flat list of chunk dicts, each carrying
    source metadata needed for citation-style grounding later on.
    """
    all_chunks = []
    for doc in documents:
        for page_num, page_text in enumerate(doc["page_texts"], start=1):
            if not page_text.strip():
                continue
            page_chunks = text_splitter.split_text(page_text)
            for chunk_idx, chunk_text in enumerate(page_chunks):
                all_chunks.append({
                    "id": f"{doc['source']}::p{page_num}::c{chunk_idx}",
                    "text": chunk_text,
                    "source": doc["source"],
                    "page": page_num,
                    "chunk_index": chunk_idx,
                })
    return all_chunks


chunks = chunk_documents(documents)

chunk_lengths = [len(c["text"]) for c in chunks]
print(f"Total chunks created: {len(chunks)}")
if chunk_lengths:
    print(f"Avg chunk length:     {sum(chunk_lengths) / len(chunk_lengths):.0f} chars")
    print(f"Min / Max length:     {min(chunk_lengths)} / {max(chunk_lengths)} chars")

# Preview first chunk
if chunks:
    print("\n--- Sample chunk ---")
    print(chunks[0]["id"])
    print(chunks[0]["text"][:300], "...")


Total chunks created: 47
Avg chunk length:     683 chars
Min / Max length:     168 / 799 chars

--- Sample chunk ---
AI-info.pdf::p1::c0
The History of Artificial
Intelligence
Artificial Intelligence (AI) is one of the most important technological developments in
modern history. It refers to the ability of machines and computer systems to perform
tasks that normally require human intelligence, such as learning, problem-solving,
under ...


## 3. Embeddings & Vector Store

We embed each chunk locally with `sentence-transformers/all-MiniLM-L6-v2` (384-dim, fast, strong general-purpose retrieval performance for its size) and store vectors + text + metadata in a **persistent ChromaDB collection** under `data/vector_store/`, so the backend can load it later without recomputing anything.

In [5]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension:    {embedding_model.get_sentence_embedding_dimension()}")


Loaded embedding model: all-MiniLM-L6-v2
Embedding dimension:    384


In [6]:
# ---- Persistent ChromaDB client, stored inside data/vector_store/ ----
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR),
    settings=Settings(anonymized_telemetry=False),
)

COLLECTION_NAME = "document_assistant"

# Fresh run: drop any pre-existing collection with the same name so re-running
# this notebook doesn't duplicate vectors.
existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"embedding_model": EMBEDDING_MODEL_NAME, "hnsw:space": "cosine"},
)

print(f"Created ChromaDB collection '{COLLECTION_NAME}' at {VECTOR_STORE_DIR}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Created ChromaDB collection 'document_assistant' at F:\rag-assistant-project\backend\data\vector_store


In [7]:
def embed_and_store(chunks, collection, embedding_model, batch_size=64):
    """Embed chunks in batches and upsert them into the Chroma collection."""
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["id"] for c in batch]
        metadatas = [
            {"source": c["source"], "page": c["page"], "chunk_index": c["chunk_index"]}
            for c in batch
        ]
        vectors = embedding_model.encode(texts, show_progress_bar=False).tolist()

        collection.upsert(
            ids=ids,
            embeddings=vectors,
            documents=texts,
            metadatas=metadatas,
        )
    return collection.count()


t0 = time.time()
stored_count = embed_and_store(chunks, collection, embedding_model)
print(f"Stored {stored_count} chunks in ChromaDB in {time.time() - t0:.1f}s")


Stored 47 chunks in ChromaDB in 1.7s


## 4. Retrieval & Prompting

`retrieve()` embeds the user's question with the same model and pulls the top-k most similar chunks from Chroma. `build_prompt()` then assembles a grounded prompt that:

- Instructs the model to answer **only** from the provided context
- Numbers each context snippet so the model can cite `[1]`, `[2]`, etc.
- Returns the source metadata (file + page) alongside the generated answer, so every claim can be traced back to its origin.

In [8]:
def retrieve(query: str, top_k: int = 4):
    """Return the top_k most relevant chunks (with metadata + distance) for a query."""
    query_vector = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=top_k)

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"],
            "distance": results["distances"][0][i],
        })
    return retrieved


PROMPT_TEMPLATE = """You are a document assistant. Answer the question using ONLY the context snippets below.
If the answer is not contained in the context, say "I don't have enough information in the provided documents to answer that."
Cite the snippet number(s) you used, like [1] or [1][3].

Context:
{context_block}

Question: {question}

Answer (with citations):"""


def build_prompt(question: str, retrieved_chunks: list) -> str:
    context_block = "\n\n".join(
        f"[{i+1}] (source: {c['source']}, page {c['page']})\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    return PROMPT_TEMPLATE.format(context_block=context_block, question=question)


In [9]:
def generate_answer(question: str, top_k: int = 4):
    """
    Full retrieve -> prompt -> generate pipeline.

    Generation is pluggable: if ANTHROPIC_API_KEY is set in the environment,
    this calls the Claude API. Otherwise it falls back to a clearly-labelled
    stub so the pipeline still runs end-to-end without an API key configured.
    Swap in your LLM provider of choice here for the real assignment submission.
    """
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}],
        )
        answer_text = response.content[0].text
    else:
        # Fallback stub: no LLM configured, so we surface the top chunk as a
        # transparent placeholder rather than fabricating a generated answer.
        top = retrieved_chunks[0] if retrieved_chunks else None
        answer_text = (
            "[No LLM configured — set ANTHROPIC_API_KEY or wire in your provider "
            "of choice inside generate_answer(). Showing top retrieved snippet instead.]\n\n"
            + (top["text"][:400] if top else "No relevant chunks found.")
        )

    sources = [{"source": c["source"], "page": c["page"]} for c in retrieved_chunks]
    return {
        "question": question,
        "answer": answer_text,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks,
        "prompt": prompt,
    }


## 5. Evaluation

We run a handful of sample questions through the full pipeline and lay out the results (retrieved sources + answer) in a table for quick manual review.

In [10]:
# Edit these to match your actual document content for a meaningful eval
SAMPLE_QUESTIONS = [
    "What is the main topic of this document?",
    "Summarize the key points in one paragraph.",
    "Does the document mention any dates or deadlines?",
]

eval_rows = []
for q in SAMPLE_QUESTIONS:
    result = generate_answer(q, top_k=3)
    source_str = "; ".join(f"{s['source']} (p.{s['page']})" for s in result["sources"])
    eval_rows.append({
        "question": q,
        "top_sources": source_str,
        "num_chunks_retrieved": len(result["retrieved_chunks"]),
        "answer_preview": result["answer"][:200].replace("\n", " ") + ("..." if len(result["answer"]) > 200 else ""),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


,question,top_sources,num_chunks_retrieved,answer_preview
0,What is the main topic of this document?,AI-info.pdf (p.3); AI-info.pdf (p.10); AI-info...,3,[No LLM configured — set ANTHROPIC_API_KEY or ...
1,Summarize the key points in one paragraph.,AI-info.pdf (p.6); AI-info.pdf (p.6); AI-info....,3,[No LLM configured — set ANTHROPIC_API_KEY or ...
2,Does the document mention any dates or deadlines?,AI-info.pdf (p.1); AI-info.pdf (p.10); AI-info...,3,[No LLM configured — set ANTHROPIC_API_KEY or ...


**Reading the results table:**
- `top_sources` — confirms retrieval is pulling from the right document/page (sanity check for grounding).
- `num_chunks_retrieved` — should equal `top_k` unless the store has fewer chunks than that.
- `answer_preview` — truncated generated answer; if `ANTHROPIC_API_KEY` isn't set, this will show the fallback stub instead of a real generation — that's expected and just means the retrieval half of the pipeline is what's being validated here.

## 6. Export — Confirm Persistence

The `PersistentClient` above already writes to disk on every `upsert`. This cell just re-opens the store fresh (simulating what the backend will do) to confirm it loads correctly without rebuilding.

In [11]:
# Simulate a fresh process loading the persisted store (what the backend will do)
verify_client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR),
    settings=Settings(anonymized_telemetry=False),
)
verify_collection = verify_client.get_collection(COLLECTION_NAME)

print(f"Reloaded collection '{COLLECTION_NAME}' from disk: {verify_collection.count()} chunks")
print(f"Vector store persisted at: {VECTOR_STORE_DIR}")

assert verify_collection.count() == stored_count, "Mismatch between stored and reloaded chunk counts!"
print("\n✅ Vector store verified — backend can load this directly via:")
print(f'   chromadb.PersistentClient(path="{VECTOR_STORE_DIR}").get_collection("{COLLECTION_NAME}")')


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Reloaded collection 'document_assistant' from disk: 47 chunks
Vector store persisted at: F:\rag-assistant-project\backend\data\vector_store

✅ Vector store verified — backend can load this directly via:
   chromadb.PersistentClient(path="F:\rag-assistant-project\backend\data\vector_store").get_collection("document_assistant")


---
### Phase 2 Checklist

- [x] Loaded & inspected documents from `data/raw/` with format/page/error reporting
- [x] Chunked with `RecursiveCharacterTextSplitter` (justified `chunk_size=800`, `chunk_overlap=120`)
- [x] Generated embeddings with `all-MiniLM-L6-v2` and persisted them in ChromaDB at `data/vector_store/`
- [x] Built a retrieval function + grounded prompt template with source citations
- [x] Ran sample-question evaluation and produced a results table
- [x] Verified the vector store reloads from disk without rebuilding

**Next (Phase 3 candidate):** wire `generate_answer()` to your chosen LLM provider for real end-to-end generation, and move retrieval/prompt logic into a reusable module the backend service can import.